In [1]:
import pandas as pd

link_stream = pd.read_csv('data/CollegeMsg.txt', delimiter=' ', names=['source', 'destination', 'timestamp'])

In [2]:
import pandas as pd


nodes = pd.concat([link_stream['source'], link_stream['destination']]).unique()

node2id = {node: idx for idx, node in enumerate(nodes)}

link_stream['source'] = link_stream['source'].map(node2id)
link_stream['destination'] = link_stream['destination'].map(node2id)
link_stream['idx'] = range(len(link_stream))

In [3]:
link_stream.to_csv('data/CollegeMsg.csv', index=False)

In [4]:
import pandas as pd

link_stream = pd.read_csv('data/CollegeMsg.csv')

In [8]:
import torch
import pandas as pd
import numpy as np


class OfflineStateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0

        self.node_time_pos = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.node_time_indptr = None   # shape: [num_nodes + 1], int64
        self.node_time_values = None   # shape: [total_appearances], int32/int64 (timestamps)



    def pre_scan_data(self, link_stream):
        self.m = float(len(link_stream))
        print(f"Total links: {int(self.m)}")

        all_nodes = pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        t_max = link_stream['timestamp'].max()
        t_min = link_stream['timestamp'].min()
        total_duration = t_max - t_min
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"T_duration = {total_duration}")
    
        # compute node lifespans
        sources = link_stream[['source', 'timestamp']].rename(columns={'source': 'node'})
        destinations = link_stream[['destination', 'timestamp']].rename(columns={'destination': 'node'})
        all_events = pd.concat([sources, destinations], ignore_index=True)
        
        node_stats = all_events.groupby('node')['timestamp'].agg(['min', 'max'])
        
        epsilon = 1.0 
        lifespans = (node_stats['max'] - node_stats['min']).clip(lower=epsilon)
        
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        
        self.node_lifespans.fill_(epsilon)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")

        # Sort by (node, timestamp). Use stable sort to keep deterministic order for same timestamps.
        all_events_sorted = all_events.sort_values(['node', 'timestamp'], kind='mergesort')

        nodes_np = all_events_sorted['node'].to_numpy(dtype=np.int64, copy=False)
        times_np = all_events_sorted['timestamp'].to_numpy(copy=False)

        if np.issubdtype(times_np.dtype, np.integer):
            # choose int32 if safe
            if times_np.min() >= np.iinfo(np.int32).min and times_np.max() <= np.iinfo(np.int32).max:
                times_np = times_np.astype(np.int32, copy=False)
            else:
                times_np = times_np.astype(np.int64, copy=False)
        else:
            # float timestamps: keep float32/float64; typically float32 is enough
            times_np = times_np.astype(np.float32, copy=False)

        # Count appearances per node
        counts = np.bincount(nodes_np, minlength=self.num_nodes).astype(np.int64)  # length n
        indptr = np.empty(self.num_nodes + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(counts, out=indptr[1:])

        # Save to torch (CPU/GPU)
        self.node_time_indptr = torch.from_numpy(indptr).to(self.device)
        self.node_time_values = torch.from_numpy(times_np).to(self.device)

        print(f"Total node appearances stored: {self.node_time_values.numel()} (expected ~ {2*int(self.m)})")

    def get_node_times(self, u: int) -> torch.Tensor:
        """Return 1D tensor of all appearance timestamps of node u (sorted)."""
        start = int(self.node_time_indptr[u].item())
        end = int(self.node_time_indptr[u + 1].item())
        return self.node_time_values[start:end]

    def get_next_time(self, u: int, j: int):
        """Return (t_j, t_{j+1}) for node u's j-th appearance. Raises if j+1 out of range."""
        start = int(self.node_time_indptr[u].item())
        end = int(self.node_time_indptr[u + 1].item())
        if start + j + 1 >= end:
            return None, None
        t_j = self.node_time_values[start + j]
        t_next = self.node_time_values[start + j + 1]
        return t_j, t_next
    
    def get_current_interval(self, u: int):
        """
        O(1) 返回当前出现时间 t1、下一次出现时间 t2，以及 delta = t2 - t1
        如果这是最后一次出现，返回 delta=0
        """
        u = int(u)
        pos = int(self.node_time_pos[u].item())

        start = int(self.node_time_indptr[u].item())
        end   = int(self.node_time_indptr[u + 1].item())

        idx = start + pos

        # Safety check
        if idx >= end:
            # Should not normally happen
            return None, None, 0.0

        t1 = self.node_time_values[idx]

        if idx + 1 < end:
            t2 = self.node_time_values[idx + 1]
            delta = float(t2 - t1)
        else:
            # last appearance: no future interval
            t2 = None
            delta = 0.0

        return t1, t2, delta
    
    def advance_time_pos(self, u: int):
        self.node_time_pos[u] += 1


In [4]:
import torch

class GlobalDegreeSampler:
    def __init__(self, global_degrees):
        """
        初始化采样器
        global_degrees: Tensor [N], 也就是 state_mgr.global_degree
        """
        self.num_nodes = len(global_degrees)
        self.degrees = global_degrees.float()
        self.weights = self.degrees

    def sample(self, num_samples):
        """
        采样 num_samples 个负样本节点 ID
        返回: Tensor [num_samples]
        """
        # torch.multinomial 实现了加权无放回/有放回采样
        # replacement=True: 允许重复采样（在大图中通常没问题且更快）
        neg_samples = torch.multinomial(self.weights, num_samples, replacement=True)
        return neg_samples

In [21]:
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 5
state_mgr = OfflineStateManager(num_nodes, num_comms, device='cpu')
state_mgr.pre_scan_data(link_stream)


num_nodes = 1899 
Total links: 59835
T_duration = 16736181
Max Lifespan: 16625344.0
Total node appearances stored: 119670 (expected ~ 119670)


In [7]:
global_sampler = GlobalDegreeSampler(state_mgr.global_degree)

In [11]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class LinkStreamDataset(Dataset):
    def __init__(self, df):
        """
        df: 包含 u, v, t 的 DataFrame。
        """
        # 1. 存数据 (Numpy 格式，极快)
        self.src = df['source'].values.astype(np.int64)
        self.dst = df['destination'].values.astype(np.int64)
        self.times = df['timestamp' ].values.astype(np.float32)
        self.edge_idxs = df.index.values.astype(np.int64)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.dst[idx], self.times[idx], self.edge_idxs[idx]

def tgn_collate_fn(batch):
    """
    自定义整理函数：把 [(u1,v1,t1,i1), (u2,v2,t2,i2)] 变成 4 个 Tensor
    """
    # zip(*batch) 会把 list of tuples 转置为 tuple of lists
    src, dst, t, edge_idx = zip(*batch)
    
    return (
        torch.tensor(src, dtype=torch.long),
        torch.tensor(dst, dtype=torch.long),
        torch.tensor(t, dtype=torch.float32),
        torch.tensor(edge_idx, dtype=torch.long)
    )

dataset = LinkStreamDataset(link_stream)
dataloader = DataLoader(dataset, batch_size=500, shuffle=False, collate_fn=tgn_collate_fn)

In [12]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path


NUM_EPOCHS = 3
BATCH_SIZE = 100
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'mps'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [14]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'CollegeMsg'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/CollegeMsg.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/CollegeMsg_node.npy), use zero vector(dim=16)...
The dataset has 59835 interactions, involving 1899 different nodes


In [15]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [16]:
from tgn.utils.my_data import get_data, compute_time_statistics
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= \
  compute_time_statistics(
      full_data.sources, 
      full_data.destinations, 
      full_data.timestamps
  )


In [22]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path




NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'mps'


aggregator = 'last'
memory_update_at_end = True
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'college_msg'

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,

    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 

).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [ ]:
import math 
import logging
import time


BATCH_SIZE = 20

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 120


In [ ]:
NUM_EPOCHS = 1


for epoch in range(NUM_EPOCHS):
    start_epoch_time = time.time()

    if USE_MEMORY:
        tgn.memory.reset_state()
    
    tgn.set_neighbor_finder(ngh_finder)
    m_loss = []
    logger.info(f'Starting epoch {epoch} / {NUM_EPOCHS} ')
    for k in range(0, num_batches):
        loss = 0
        optimizer.zero_grad()

        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))
        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]

        size = len(sources_batch)


        tgn = tgn.train()
        

In [ ]:
import pandas as pd
import torch
import numpy as np
import os

def export_community_structure(tgn, state_mgr, dataloader, device, output_path="results/community_structure.csv"):
    """
    执行推理，并将结果导出为 CSV
    格式: u, v, timestamp, u_comm, v_comm
    """
    print(f"开始导出社区结构到: {output_path}")
    
    # 1. 切换到评估模式
    tgn.eval()
    
    # 2. (可选) 重置状态 - 如果你希望输出的是基于最终训练好的模型的"完美回放"
    # 如果你想保留训练结束时的状态，注释掉下面两行
    state_mgr.on_epoch_start() 
    if tgn.use_memory:
        tgn.memory.__init_memory__()
        
    # 结果列表
    results = []
    
    # 3. 不计算梯度，节省显存
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            # 解包
            src, dst, t, edge_idxs = batch
            src = src.to(device)
            dst = dst.to(device)
            t = t.to(device)
            
            # A. 前向传播 (计算 Embedding)
            z_src = tgn.compute_temporal_embeddings(src, t)
            z_dst = tgn.compute_temporal_embeddings(dst, t)
            
            # B. 计算社区概率 (Probabilities)
            p_src = tgn.community_projector(z_src) # [B, K]
            p_dst = tgn.community_projector(z_dst) # [B, K]
            
            # C. 转换为社区 ID (Argmax)
            # dim=1 表示在社区维度上找最大值索引
            comm_src = torch.argmax(p_src, dim=1).cpu().numpy()
            comm_dst = torch.argmax(p_dst, dim=1).cpu().numpy()
            
            # D. 转回 CPU 以便存储
            src_cpu = src.cpu().numpy()
            dst_cpu = dst.cpu().numpy()
            t_cpu = t.cpu().numpy()
            
            # E. 收集结果
            for i in range(len(src)):
                results.append({
                    "u": src_cpu[i],
                    "v": dst_cpu[i],
                    "timestamp": t_cpu[i],
                    "u_comm": comm_src[i],
                    "v_comm": comm_dst[i]
                })
            
            # F. 别忘了更新 Memory！(推理时也需要更新记忆，否则后续的时间步无法利用之前的交互信息)
            if tgn.use_memory:
                uniq_s, msg_s = tgn.get_raw_messages(src, z_src, dst, z_dst, t, edge_idxs)
                uniq_d, msg_d = tgn.get_raw_messages(dst, z_dst, src, z_src, t, edge_idxs)
                tgn.memory.store_raw_messages(uniq_s, msg_s)
                tgn.memory.store_raw_messages(uniq_d, msg_d)
                tgn.memory.detach_memory() # 虽然是 no_grad，但加上保险
                
            # (可选) 打印进度
            if batch_idx % 100 == 0:
                print(f"Processing batch {batch_idx}...")

    # 4. 写入 CSV
    df = pd.DataFrame(results)
    
    # 确保目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    df.to_csv(output_path, index=False)
    print("导出完成！")
    return df

export_community_structure(tgn, state_mgr, dataloader, device, "results/final_communities.csv")

开始导出社区结构到: results/final_communities.csv
torch.cuda.FloatTensor
torch.cuda.FloatTensor
Processing batch 0...
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatT

,u,v,timestamp,u_comm,v_comm
0,0,824,0.0,2,2
1,1,216,2797.0,2,2
2,1,72,3304.0,2,2
3,2,152,4523.0,2,2
4,2,332,7926.0,2,2
...,...,...,...,...,...
332329,357,790,45401816.0,2,2
332330,152,208,45402440.0,2,2
332331,152,208,45403708.0,2,2
332332,342,208,45404904.0,2,2


: 